# Phase D — GPU Retrain (Kaggle / Colab compatible)

Cross-platform notebook for the v3 cascade train. Auto-detects host (Kaggle / Colab / local).

**Hardware**: T4 (Kaggle) or T4/A100 (Colab Pro). Free Kaggle = 30 hr/wk T4×2.

**Pause/resume**: every 500 train steps writes `target/<stage>-v3-gpu/checkpoint-N/`. Re-running the same train cell auto-resumes from the latest checkpoint.

**Total runtime estimate (T4)**: ~5-7 hr.
  - Install + data fetch: ~15 min
  - Corpus build: ~15-20 min
  - Stage 1 linker: ~1 hr
  - Stage 3 slot-filler: ~1 hr
  - Stage 2 skeleton (50K JOIN-balanced × 3 ep): ~2-3 hr
  - Export: ~5 min

In [ ]:
# Cell 1 — env detect + repo clone
import os, subprocess, sys

# Auto-detect platform working dir.
if os.path.exists('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    WORK_DIR = '/content'
else:
    WORK_DIR = os.path.expanduser('~/db-claw-work')
    os.makedirs(WORK_DIR, exist_ok=True)

REPO_ROOT = f'{WORK_DIR}/db-claw'
DATA_DIR = f'{REPO_ROOT}/data'
TARGET_DIR = f'{REPO_ROOT}/target'

os.chdir(WORK_DIR)
if os.path.exists(REPO_ROOT):
    print(f'{REPO_ROOT} exists — pulling latest')
    subprocess.check_call(['git', '-C', REPO_ROOT, 'pull', '--rebase'])
else:
    subprocess.check_call(['git', 'clone', 'https://github.com/xwiz/db-claw.git', REPO_ROOT])

os.chdir(REPO_ROOT)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(TARGET_DIR, exist_ok=True)

print(f'WORK_DIR={WORK_DIR}')
print(f'REPO_ROOT={REPO_ROOT}')
print(f'cwd={os.getcwd()}')
subprocess.check_call(['git', 'log', '--oneline', '-5'])

In [ ]:
# Cell 2 — install deps + verify GPU
%pip install -q -e python/semsql_train python/semsql_eval python/semsql_rewriter
%pip install -q transformers==4.45.2 datasets accelerate optimum[onnxruntime] sqlglot click pyarrow huggingface_hub sentencepiece onnx onnxruntime

# Refresh import paths after install.
import importlib, site
importlib.invalidate_caches()
site.main()

import torch
assert torch.cuda.is_available(), 'GPU required — switch runtime to T4'
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} dev={torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3 — fetch raw HF corpora (~7-8 GB)
from huggingface_hub import hf_hub_download, snapshot_download
import shutil, os

os.makedirs(f'{DATA_DIR}', exist_ok=True)

# SQaLe — 33 parquet shards, biggest contributor
snapshot_download(
    repo_id='trl-lab/SQaLe-text-to-SQL-dataset',
    repo_type='dataset',
    local_dir=f'{DATA_DIR}/sqale',
    allow_patterns=['data/*.parquet'],
)

# NSText2SQL — 433 MB jsonl
p = hf_hub_download(repo_id='NumbersStation/NSText2SQL', filename='train.jsonl', repo_type='dataset')
shutil.copy(p, f'{DATA_DIR}/nstext2sql_train.jsonl')

# Gretel synthetic — 32 MB parquet
p = hf_hub_download(repo_id='gretelai/synthetic_text_to_sql', filename='synthetic_text_to_sql_train.snappy.parquet', repo_type='dataset')
shutil.copy(p, f'{DATA_DIR}/gretel_synth_text_to_sql_train.parquet')

# OmniSQL-BIRD — 80 MB json
p = hf_hub_download(repo_id='xxxbrem/OmniSQL-BIRD', filename='train_bird.json', repo_type='dataset')
shutil.copy(p, f'{DATA_DIR}/omnisql_bird_train.json')

import glob
sqale_shards = glob.glob(f'{DATA_DIR}/sqale/data/*.parquet')
print(f'SQaLe shards: {len(sqale_shards)}')
print(f"data dir size: {sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DATA_DIR) for f in fs) / 1e9:.2f} GB")

In [ ]:
# Cell 4 — build all 4 v3 corpora via direct Python imports
import sys, importlib.util
from pathlib import Path
import glob

# Manual import of teacher_cache (avoids any pip-install path edge cases)
module_path = f'{REPO_ROOT}/python/semsql_train/src/semsql_train/teacher_cache.py'
spec = importlib.util.spec_from_file_location('semsql_train.teacher_cache', module_path)
tc = importlib.util.module_from_spec(spec)
sys.modules['semsql_train.teacher_cache'] = tc
spec.loader.exec_module(tc)

sqale_shards = sorted(glob.glob(f'{DATA_DIR}/sqale/data/*.parquet'))
assert sqale_shards, 'SQaLe parquet shards missing — re-run cell 3'
sqale_glob = sqale_shards[0].rsplit('/', 1)[0] + '/*.parquet'

s1 = tc.build_teacher_cache_from_sqale(
    out_jsonl=Path(f'{DATA_DIR}/skeleton_train_v3_full.jsonl'),
    parquet_glob=sqale_glob,
)
print(f'sqale: {s1.converted}/{s1.total} retention={s1.retention:.1%}')

s2 = tc.build_teacher_cache_from_nstext2sql_jsonl(
    in_jsonl=Path(f'{DATA_DIR}/nstext2sql_train.jsonl'),
    out_jsonl=Path(f'{DATA_DIR}/skeleton_train_v3_nstext2sql.jsonl'),
)
print(f'nstext2sql: {s2.converted}/{s2.total} retention={s2.retention:.1%}')

s3 = tc.build_teacher_cache_from_omnisql(
    out_jsonl=Path(f'{DATA_DIR}/skeleton_train_v3_gretel.jsonl'),
    parquet_glob=f'{DATA_DIR}/gretel_synth_text_to_sql_train.parquet',
    question_key='sql_prompt',
    sql_key='sql',
    db_id_key='domain',
)
print(f'gretel: {s3.converted}/{s3.total} retention={s3.retention:.1%}')

s4 = tc.build_teacher_cache_from_omnisql_bird_json(
    in_json=Path(f'{DATA_DIR}/omnisql_bird_train.json'),
    out_jsonl=Path(f'{DATA_DIR}/skeleton_train_v3_omnisql_bird.jsonl'),
)
print(f'omnisql-bird: {s4.converted}/{s4.total} retention={s4.retention:.1%}')

In [ ]:
# Cell 5 — concat ultimate corpus + JOIN-balanced 50K subset for Stage 2
import os, random, subprocess

ultimate = f'{DATA_DIR}/skeleton_train_v3_ultimate.jsonl'
subprocess.check_call(
    f'cat {DATA_DIR}/skeleton_train_v3_full.jsonl '
    f'{DATA_DIR}/skeleton_train_v3_nstext2sql.jsonl '
    f'{DATA_DIR}/skeleton_train_v3_gretel.jsonl '
    f'{DATA_DIR}/skeleton_train_v3_omnisql_bird.jsonl '
    f'> {ultimate}',
    shell=True,
)
with open(ultimate) as f: lines = f.readlines()
join_lines = [l for l in lines if 'INNER JOIN' in l]
fk_lines = [l for l in lines if '"kind": "fk"' in l]
print(f'ultimate: {len(lines)} rows | JOINs: {len(join_lines)} | FK: {len(fk_lines)}')

# Build JOIN-balanced 50K subset.
random.seed(0xCAFEF00D)
nojoin_lines = [l for l in lines if 'INNER JOIN' not in l]
random.shuffle(join_lines); random.shuffle(nojoin_lines)
sub = join_lines[:25000] + nojoin_lines[:25000]
random.shuffle(sub)
with open(f'{DATA_DIR}/skeleton_train_v3_50k_balanced.jsonl', 'w') as f:
    f.writelines(sub)
print(f'50K subset: {len(sub)} ({sum(1 for l in sub if "INNER JOIN" in l)} JOINs)')

# Tiny eval split for the trainer's preflight (not used for measurement).
with open(f'{DATA_DIR}/skeleton_eval.jsonl', 'w') as f:
    f.writelines(sub[:500])
print(f'skeleton eval: 500 rows (subset of 50K)')

In [ ]:
# Cell 6 — derive Stage 1 linker pairs + Stage 3 slot pairs
import subprocess, random, os

subprocess.check_call([
    'python', '-m', 'semsql_train', 'derive-linker-pairs',
    '--in', f'{DATA_DIR}/skeleton_train_v3_ultimate.jsonl',
    '--out', f'{DATA_DIR}/linker_train_v3.jsonl',
    '--max-rows', '200000',
], cwd=REPO_ROOT)

random.seed(7)
with open(f'{DATA_DIR}/linker_train_v3.jsonl') as f: lines = f.readlines()
random.shuffle(lines)
with open(f'{DATA_DIR}/linker_eval_v3.jsonl', 'w') as f: f.writelines(lines[:8000])
with open(f'{DATA_DIR}/linker_train_v3.jsonl', 'w') as f: f.writelines(lines[8000:])
print(f'linker train={len(lines)-8000} eval=8000')

subprocess.check_call([
    'python', '-m', 'semsql_train', 'derive-slot-pairs',
    '--in', f'{DATA_DIR}/skeleton_train_v3_ultimate.jsonl',
    '--out', f'{DATA_DIR}/slot_train_v3.jsonl',
    '--max-rows', '200000',
], cwd=REPO_ROOT)

with open(f'{DATA_DIR}/slot_train_v3.jsonl') as f: lines = f.readlines()
random.shuffle(lines)
with open(f'{DATA_DIR}/slot_eval_v3.jsonl', 'w') as f: f.writelines(lines[:8000])
with open(f'{DATA_DIR}/slot_train_v3.jsonl', 'w') as f: f.writelines(lines[8000:])
print(f'slot train={len(lines)-8000} eval=8000')

In [ ]:
# Cell 7 — train Stage 1 linker (~1 hr T4). Auto-resumes from latest checkpoint.
import subprocess
subprocess.check_call([
    'python', '-m', 'semsql_train', 'train',
    '--stage', 'linker',
    '--base-model', 'distilbert-base-uncased',
    '--train', f'{DATA_DIR}/linker_train_v3.jsonl',
    '--eval', f'{DATA_DIR}/linker_eval_v3.jsonl',
    '--epochs', '1',
    '--batch-size', '64',
    '--out', f'{TARGET_DIR}/linker-v3-gpu',
], cwd=REPO_ROOT)

In [ ]:
# Cell 8 — train Stage 3 slot filler (~1 hr T4). Auto-resumes.
import subprocess
subprocess.check_call([
    'python', '-m', 'semsql_train', 'train',
    '--stage', 'slot-filler',
    '--base-model', 'distilbert-base-uncased',
    '--train', f'{DATA_DIR}/slot_train_v3.jsonl',
    '--eval', f'{DATA_DIR}/slot_eval_v3.jsonl',
    '--epochs', '1',
    '--batch-size', '64',
    '--out', f'{TARGET_DIR}/slot-filler-v3-gpu',
], cwd=REPO_ROOT)

In [ ]:
# Cell 9 — train Stage 2 skeleton (50K JOIN-balanced × 3 ep, ~2-3 hr T4). Auto-resumes.
import subprocess
subprocess.check_call([
    'python', '-m', 'semsql_train', 'train',
    '--stage', 'skeleton',
    '--base-model', 't5-small',
    '--train', f'{DATA_DIR}/skeleton_train_v3_50k_balanced.jsonl',
    '--eval', f'{DATA_DIR}/skeleton_eval.jsonl',
    '--epochs', '1',
    '--batch-size', '16',
    '--grad-accum', '2',
    '--out', f'{TARGET_DIR}/skeleton-v3-gpu',
], cwd=REPO_ROOT)

In [ ]:
# Cell 10 — export cascade + tarball
import subprocess, json, shutil, os
import torch
if not hasattr(torch, 'int4'):
    torch.int4 = torch.qint8 if hasattr(torch, 'qint8') else torch.int8

cascade_dir = f'{TARGET_DIR}/cascade-v3-gpu'
os.makedirs(cascade_dir, exist_ok=True)

subprocess.check_call([
    'python', '-m', 'semsql_train', 'export-cascade',
    '--output-dir', cascade_dir,
    '--cascade-version', 'v0.3.0-gpu',
    '--skeleton-checkpoint', f'{TARGET_DIR}/skeleton-v3-gpu',
    '--linker-checkpoint', f'{TARGET_DIR}/linker-v3-gpu',
    '--slot-filler-checkpoint', f'{TARGET_DIR}/slot-filler-v3-gpu',
    '--no-int8',  # avoid torch.int4 path on older torch
], cwd=REPO_ROOT)

# Patch manifest paths.
with open(f'{cascade_dir}/manifest.json') as f: m = json.load(f)
m['skeleton']['path'] = '_skeleton_export'
m['skeleton']['tokenizer'] = '_skeleton_export/tokenizer.json'
m['linker']['path'] = '_linker_export/model.onnx'
m['linker']['tokenizer'] = '_linker_export/tokenizer.json'
m['slot_filler']['path'] = '_slot_filler_export/model.onnx'
m['slot_filler']['tokenizer'] = '_slot_filler_export/tokenizer.json'
with open(f'{cascade_dir}/manifest.json', 'w') as f: json.dump(m, f, indent=2)
print('manifest patched')

# Tarball — Kaggle outputs go to /kaggle/working (auto-downloadable from Output panel).
tar_path = f'{WORK_DIR}/cascade-v3-gpu.tar.gz'
subprocess.check_call(['tar', 'czf', tar_path, '-C', TARGET_DIR, 'cascade-v3-gpu'])
print(f'tarball: {tar_path} ({os.path.getsize(tar_path) / 1e6:.1f} MB)')
print('Done. Download from the Kaggle Output panel (right side bar).')
